In [5]:
import re
import json
import csv

with open("additionalQ402.txt", "r", encoding="utf-8") as f:
    content = f.read()

def normalize_digits(text):
    persian = "۰۱۲۳۴۵۶۷۸۹"
    arabic  = "٠١٢٣٤٥٦٧٨٩"
    latin   = "0123456789"
    trans = str.maketrans(persian + arabic, latin * 2)
    return text.translate(trans)

content = normalize_digits(content)
lines = content.split("\n")

# ===== تنها تغییر مهم: فقط exact match با لیست مشخص =====
KNOWN_CATEGORIES = [
    "حقوق مدنی",
    "حقوق تجارت",
    "حقوق جزای عمومی",
    "حقوق جزای اختصاصی",
    "حقوق جزا",
    "آیین دادرسی کیفری",
    "آیین دادرسی مدنی",
    "حقوق اساسی",
    "حقوق بین الملل",
    "حقوق بین‌الملل",
]

def is_category_line(line):
    """فقط اگر خط دقیقاً با یکی از category‌های شناخته‌شده برابر بود"""
    stripped = line.strip()
    return stripped in KNOWN_CATEGORIES

# ============================================================

category = "نامشخص"
questions = []
current_q = None
current_opts = []

for line in lines:
    line_stripped = line.strip()
    if not line_stripped:
        continue

    opt_match = re.match(r'^\(?([1-4])\)?[\)\.\-:]\s*(.+)$', line_stripped)
    q_match   = re.match(r'^(\d+)\s*[-–\.):]?\s*(.+)$', line_stripped)

    # --- تشخیص category (فقط exact match) ---
    if is_category_line(line_stripped):
        if current_q is not None:
            questions.append({
                "question_number": current_q["number"],
                "category":        current_q["category"],
                "question":        current_q["text"],
                "options":         current_opts
            })
            current_q    = None
            current_opts = []
        category = line_stripped
        continue

    # --- سوال جدید ---
    if q_match and not opt_match:
        qnum = int(q_match.group(1))
        if 1 <= qnum <= 140:
            if current_q is not None:
                questions.append({
                    "question_number": current_q["number"],
                    "category":        current_q["category"],
                    "question":        current_q["text"],
                    "options":         current_opts
                })
            current_q = {
                "number":   str(qnum),
                "category": category,
                "text":     q_match.group(2).strip()
            }
            current_opts = []
        # اعداد خارج از بازه (شماره صفحه و ...) نادیده گرفته می‌شوند

    # --- گزینه ---
    elif opt_match and current_q is not None:
        current_opts.append(f"{opt_match.group(1)}) {opt_match.group(2).strip()}")

    # --- ادامه متن سوال یا گزینه ---
    elif current_q is not None:
        if not current_opts:
            current_q["text"] += " " + line_stripped
        else:
            current_opts[-1] += " " + line_stripped

# ذخیره آخرین سوال
if current_q is not None:
    questions.append({
        "question_number": current_q["number"],
        "category":        current_q["category"],
        "question":        current_q["text"],
        "options":         current_opts
    })

questions.sort(key=lambda x: int(x["question_number"]))

with open("structured_questions_402.json", "w", encoding="utf-8") as jf:
    json.dump(questions, jf, ensure_ascii=False, indent=2)

with open("structured_questions_402.csv", "w", encoding="utf-8-sig", newline="") as cf:
    writer = csv.DictWriter(cf, fieldnames=["question_number", "category", "question", "options"])
    writer.writeheader()
    for q in questions:
        writer.writerow({
            "question_number": q["question_number"],
            "category":        q["category"],
            "question":        q["question"],
            "options":         " | ".join(q["options"])
        })

print(f"✓ تعداد سوالات: {len(questions)}")
print(f"✓ فایل‌ها ذخیره شدند: structured_questions_402.json, structured_questions_402.csv")

from collections import Counter
cats = Counter(q["category"] for q in questions)
print("\n📊 توزیع category‌ها:")
for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count} سوال")

✓ تعداد سوالات: 10
✓ فایل‌ها ذخیره شدند: structured_questions_402.json, structured_questions_402.csv

📊 توزیع category‌ها:
  نامشخص: 10 سوال
